# Check 1: Genie Precision, Recall, and the Missing Threshold

This notebook runs **Check 1** from the Genie Test Protocol end to end.

The check asks Genie a plain inbound-transfer ranking question — the same question an analyst would ask when starting a fraud investigation — and then measures how accurate that answer is against the known fraud ground truth.

**Four phases:**

- **Phase 1** — Call Genie with the Check 1 question and display the result as a table
- **Phase 2** — Confirm the `account_labels` ground truth table is accessible
- **Phase 3** — Join the result against ground truth and compute precision and recall
- **Phase 4** — Explain why threshold sensitivity cannot be measured for this result

**Before running:** Set your `SPACE_ID` in the configuration cell below. The Genie Space must be connected to the tables created by `setup/upload_and_create_tables.sh`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set SPACE_ID to your Genie Space ID before running.
# Find it in Databricks under Genie → your space → the ID in the URL.

SPACE_ID = "YOUR-GENIE-SPACE-ID"   # <-- replace this

CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"
VOLUME   = "graph-enriched-volume"

In [ ]:
import os
import sys

from databricks.sdk import WorkspaceClient
import pandas as pd

# ── Import shared demo utilities ──────────────────────────────────────────────
_DEMO_DIR = os.path.join(os.getcwd(), "finance-genie", "genie_demos")
if _DEMO_DIR not in sys.path:
    sys.path.insert(0, _DEMO_DIR)

from demo_utils import genie_caller, load_ground_truth, label_accounts

# ── Connect and bind Genie caller ─────────────────────────────────────────────
w = WorkspaceClient()
print("Connected to:", w.config.host)

ask_genie = genie_caller(w, SPACE_ID)

In [ ]:
# ── Load ground truth ────────────────────────────────────────────────────────
_GT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"
gt = load_ground_truth(_GT_PATH)
print(
    f"Ground truth: {gt['summary']['total_rings']} rings, "
    f"{gt['summary']['total_fraud_accounts']:,} fraud accounts, "
    f"{gt['summary']['total_whale_accounts']:,} whales"
)


def run_check(question: str) -> dict:
    """Ask Genie a question and compute precision and recall against ground truth.

    Reads labels_df, total_fraud, and gt from the notebook scope.
    Returns a dict with question, precision, recall, total_returned,
    true_positives, and df (enriched result DataFrame).
    """
    print(f"Question: {question}")
    print()
    response = ask_genie(question)

    if response["text"] and response["df"] is None:
        print("Genie returned a text response:")
        print(response["text"])
        return {"question": question, "precision": None, "recall": None, "df": None}

    print(f"Status: {response['status']}  |  Rows returned: {len(response['df'])}")
    print()
    print("SQL Genie generated:")
    print(response["sql"])
    print()

    # ── Identify the account ID column ────────────────────────────────────────
    id_col = next(
        (c for c in response["df"].columns if "account_id" in c.lower()),
        response["df"].columns[0],
    )
    genie_ids = (
        response["df"][[id_col]]
        .rename(columns={id_col: "account_id"})
        .assign(account_id=lambda df: df["account_id"].astype("int64"))
    )

    # ── Join is_fraud from the account_labels Delta table ─────────────────────
    joined = genie_ids.merge(labels_df, on="account_id", how="left")

    # ── Enrich with FRAUD / WHALE / NORMAL label and ring_id ──────────────────
    enriched = label_accounts(joined["account_id"].tolist(), gt)
    joined = joined.merge(
        enriched[["account_id", "label", "ring_id"]], on="account_id", how="left"
    )

    print("Genie's result with labels:")
    print()
    display(joined)

    # ── Precision and recall ──────────────────────────────────────────────────
    true_positives = int(joined["is_fraud"].sum())
    total_returned = len(joined)
    precision = true_positives / total_returned if total_returned > 0 else 0.0
    recall    = true_positives / total_fraud    if total_fraud > 0    else 0.0

    print()
    print(f"Precision: {precision:.1%}  ({true_positives} fraud in {total_returned} returned)")
    print(f"Recall:    {recall:.1%}  ({true_positives} of {total_fraud:,} known fraud accounts)")

    return {
        "question":       question,
        "precision":      precision,
        "recall":         recall,
        "total_returned": total_returned,
        "true_positives": true_positives,
        "df":             joined,
    }

## Phase 1 — Call Genie and Display Results

The question below gives Genie explicit fraud framing — asking about hubs of a money movement network that are potentially fraudulent. This is a stronger prompt than a plain inbound-count ranking, and Genie will likely surface some genuine fraud ring members alongside whale accounts.

The expected result is a mixed list: some fraud ring members from different rings, some whale accounts. The result looks more credible than a pure whale list — which makes the precision and recall analysis in Phase 3 more important, not less.

## Phase 2 — Confirm the Ground Truth Table

The `account_labels` table holds a fraud label for every account in the dataset — one row per account with `account_id` and `is_fraud`. It was loaded by `setup/upload_and_create_tables.sh` from `data/account_labels.csv`.

This table provides the denominator for recall: the total number of accounts labeled as fraud, which is the population Genie's answer is measured against.

In [ ]:
labels_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_labels").toPandas()
print(f"account_labels: {len(labels_df):,} rows\n")
display(labels_df.head(5))

In [ ]:
total_accounts = len(labels_df)
total_fraud    = int(labels_df["is_fraud"].sum())

print(f"Total accounts in dataset:  {total_accounts:,}")
print(f"Total labeled as fraud:     {total_fraud:,}")
print(f"Fraud rate:                 {total_fraud / total_accounts:.1%}")
print()
print(f"{total_fraud:,} is the recall denominator — the number of accounts")
print("Genie would need to return to achieve 100% recall.")

In [ ]:
result = run_check(
    "Are there accounts that seem to be the hub of a "
    "money movement network that are potentially fraudulent?"
)

## Phase 4 — Why Threshold Cannot Be Measured

The precision and recall numbers above tell part of the story. What matters equally — and what this section makes explicit — is that there is **no way to improve those numbers by adjusting a threshold**.

### The structural problem

Genie returned a list of accounts with no score attached to any of them. Whatever signal Genie used internally to identify these accounts as suspicious hubs — transfer patterns, network structure, or some combination — that signal was not exposed as a column. Each account simply appears in the list or does not. There is no value to cut on.

In a standard classification system, a threshold is a number you draw across a score distribution. Accounts above the threshold are flagged; accounts below are not. You can slide the threshold up or down and observe the tradeoff: higher threshold means fewer false positives (better precision) at the cost of more false negatives (worse recall). The F1 score summarizes where the tradeoff is best. This is threshold sensitivity analysis.

**None of that applies here.** There is no score to cut. There is no distribution to draw a line across. The only parameter an analyst can change is the question itself — but re-asking a different question is not threshold analysis. There is no feedback signal to tell the analyst whether a rephrased question is better or worse without running through ground truth every time.

### What an analyst actually sees

The table in Phase 1 looks like a valid answer. It is a list of account IDs Genie believes are suspicious hubs. Without external ground truth, an analyst has no way to know how accurate it is — which accounts are genuine ring members, which are high-volume legitimate accounts, and which lie somewhere in between. The result is not obviously broken. It is unmeasurably inaccurate.

The moderate precision number from Phase 3 makes this harder to see, not easier. A result that is 40–50% accurate looks like it is working. But the analyst cannot know it is 40–50% accurate without ground truth they would not have in production. And they have no lever to push it higher — no score, no cutoff, no optimization target.

### Compare to Check 4 (PageRank)

After GDS enrichment, each account has a continuous `risk_score` derived from PageRank. Ring members score high because they receive transfers from other high-scoring ring members — the centrality compounds recursively through the ring topology. Whale accounts score moderate because their senders are peripheral low-PageRank nodes.

That score is a real probability-like signal. An analyst can set a threshold of 0.7 or 0.8 and measure what happens to precision and recall at each cutoff. The tradeoff curve is visible and the analyst can pick an operating point based on their review capacity. Threshold sensitivity analysis becomes possible — and the separation between fraud and non-fraud accounts is visible without needing ground truth at all.

In [ ]:
print("=" * 62)
print("CHECK 1 SUMMARY")
print("=" * 62)
print()
print(f"Genie returned {result['total_returned']} accounts.")
print()
print(f"  Precision: {result['precision']:.1%}")
print(f"    {result['true_positives']} of {result['total_returned']} returned accounts are labeled fraud")
print()
print(f"  Recall:    {result['recall']:.1%}")
print(f"    {result['true_positives']} of {total_fraud:,} known fraud accounts were returned")
print()
print("-" * 62)
print("Threshold sensitivity: not measurable")
print("-" * 62)
print()
print("Genie's result carries no score. Every returned account")
print("appeared in the list or did not — there is no value to cut on.")
print("Without a score:")
print()
print("  - There is no cutoff to vary.")
print("  - There is no tradeoff curve to plot.")
print("  - There is no F1 score to optimize.")
print("  - An analyst cannot tell from the output alone which accounts")
print("    are fraud ring members and which are legitimate hubs.")
print()
print("The only lever available is asking a different question,")
print("which is not threshold analysis — it requires re-running the")
print("ground truth comparison every time to know if it helped.")
print()
print("-" * 62)
print("Next: Check 4 (PageRank)")
print("-" * 62)
print()
print("After GDS enrichment the accounts table gains a continuous risk_score.")
print("That score enables the analysis this result cannot support: a threshold")
print("with a measurable precision/recall tradeoff, visible without ground truth.")